# Lecture 08: Data quality and model evaluation

Five **ungraded** exercises for October 28, 2026. All students use the same
practice. Predict the result before running each calculation.

The notebook runs offline on CPU using Python's standard library. GPU memory
and total GPU-hours: **0**. Keep the `assets/` folder beside your personal copy.
The code audits values in memory and never removes source files.

**All example texts, probabilities, and model outcomes below are synthetic.**
They demonstrate evaluation methods and are not measurements of a trained model.
Sources and limitations are in [the reading guide](optional-reading.md) and
[asset provenance](assets/README.md).

In [ ]:
from pathlib import Path
import json
import random
import sys

assets = Path("assets")
assert assets.is_dir(), "Open this notebook beside its assets folder."
sys.path.insert(0, str(assets.resolve()))
from evaluation import (normalize_document, exact_duplicate_groups,
                        word_shingles, jaccard, completion_score,
                        accuracy, pass_at_k, comparison_predictions,
                        paired_bootstrap)
print("Offline CPU examples ready. All classroom outcomes are synthetic.")

## E01 · Which records belong together? · 6 minutes

Inspect the six records printed below.

1. Find exact duplicate groups after NFC normalization, line-ending conversion,
   and trimming outer whitespace. Preserve case and internal spacing.
2. Identify the group that crosses training and development splits.
3. Compare the red/blue sentences using whitespace word-bigram sets.
   Explain why similarity alone does not establish semantic equivalence.

These word shingles are **not** model BPE tokens. For a new dataset, group
related records before assigning splits. If the development set is externally
fixed, preserve it and audit training overlap using a declared policy.

In [ ]:
records = json.loads((assets / "toy-documents.json").read_text())["records"]
for record in records:
    print(record["id"], record["split"], repr(record["text"]))

Write your predicted groups here before running the audit:

- Exact groups:
- Group crossing splits:
- Similar but distinct pair:

In [ ]:
groups = exact_duplicate_groups(records)
by_id = {record["id"]: record for record in records}
for group in groups:
    splits = sorted({by_id[item]["split"] for item in group})
    print("Group:", group, "Splits:", splits)

red = word_shingles(by_id["train-1"]["text"])
blue = word_shingles(by_id["train-3"]["text"])
print("Shared bigrams:", sorted(red & blue))
print("Union size:", len(red | blue))
print("Jaccard:", jaccard(red, blue))

**Check:** `train-1/train-2` share a normalized key. NFC makes
`train-4/dev-1` equal despite different accent encodings; this group crosses
splits. The red/blue sentences share two of six distinct bigrams, so Jaccard
is `1/3`. The color can change a question's answer. No files were deleted.

## E02 · Which candidate wins under each rule? · 6 minutes

Under the same prompt, candidate A has token log probabilities `[-1.1, -1.3]`.
Candidate B has `[-0.7, -0.8, -0.8, -0.9]`. These are invented natural-log values.

1. Sum each candidate's answer-token log probabilities.
2. Divide each sum by its number of answer tokens.
3. Which candidate wins under each rule? Can you tell which rule is more accurate?

Only the answer tokens are scored. Real task definitions can use another length
unit; check the pinned configuration rather than assuming a metric name means
per-token normalization. Leading spaces and prompt/answer token boundaries matter.

In [ ]:
answers = {"A": [-1.1, -1.3], "B": [-0.7, -0.8, -0.8, -0.9]}
for per_token in (False, True):
    scores = {name: completion_score(values, per_token=per_token)
              for name, values in answers.items()}
    label = "Per token" if per_token else "Sum"
    print(label, {name: round(value, 3) for name, value in scores.items()},
          "Winner:", max(scores, key=scores.get))

**Check:** the sums are A `−2.4` and B `−3.2`, so A wins. Per-token means are
A `−1.2` and B `−0.8`, so B wins. Larger, less negative scores win.
The decision changes because the protocol changes. Without a correct task label,
this arithmetic cannot establish which scoring rule is more accurate.

## E03 · How much does the attempt budget matter? · 5 minutes

Ten sampled programs contain two that pass every supplied test. Predict pass@1
and pass@5 using

$$
\widehat{\mathrm{pass@}k}
=1-\frac{\binom{n-c}{k}}{\binom{n}{k}}.
$$

This HumanEval estimator uses `n` sampled completions, of which `c` pass, and
requires `1 ≤ k ≤ n`. The complement counts subsets containing only failures.

Does a higher pass@5 imply better model weights or equal inference cost?
Does it provide a way to choose a correct answer without a checker?

In [ ]:
n, c = 10, 2
pass1 = pass_at_k(n, c, 1)
pass5 = pass_at_k(n, c, 5)
print(f"pass@1: {pass1:.1%}")
print(f"pass@5: {pass5:.1%}")

**Check:** pass@1 is `20%`. Pass@5 is `1 − 56/252 = 7/9 ≈ 77.8%`.
The sampled outcomes and model are unchanged; the attempt budget changes.
This measure credits any successful attempt and assumes a checker. Adaptive
repair using feedback is another protocol. The classroom example executes no
model-generated programs.

Source: [Chen et al., 2021](https://arxiv.org/abs/2107.03374), Section 3 and Appendix A.

## E04 · Does the evaluation set change the conclusion? · 6 minutes

An **independent audit** marked 20 items as exposed to training. On the clean
100 items, A gets 60 correct and B gets 64. On the exposed 20, A gets 19 correct
and B gets 11. All counts are synthetic.

1. Compute both accuracies on all 120 items.
2. Compute both accuracies on the clean 100.
3. Explain what changed and what this example cannot establish.

Do not choose exclusions based on which model fails an item. A revised evaluated
population must be justified and reported.

In [ ]:
baseline, candidate = comparison_predictions()
exposed_a = [1] * 19 + [0]
exposed_b = [1] * 11 + [0] * 9
for label, a, b in [
    ("All 120", baseline + exposed_a, candidate + exposed_b),
    ("Clean 100", baseline, candidate),
]:
    print(f"{label}: A {accuracy(a):.2%}, B {accuracy(b):.2%}")

**Check:** all items give A `65.83%` and B `62.5%`; clean items give A `60%`
and B `64%`. The evaluated sample changes and the ranking reverses.
Predictions on the clean items are unchanged. This constructed example does not
estimate the causal effect of contamination in a real training run.

## E05 · How strong is the four-point improvement? · 8 minutes

The clean predictions are paired by item: 50 both correct, 10 only A correct,
14 only B correct, and 26 both wrong. The observed B−A difference is four
**percentage points**.

1. Sample 100 item indices with replacement and preserve each A/B pair.
2. Repeat 5,000 times; read the 2.5th and 97.5th percentiles.
3. State a qualified conclusion and two uncertainty sources this interval omits.

First inspect one resample, then run the helper's full calculation.

In [ ]:
differences = [b - a for a, b in zip(baseline, candidate)]
print("Observed disagreement count:", sum(value != 0 for value in differences))
print("Observed B−A (percentage points):", 100 * sum(differences) / len(differences))

rng = random.Random(20261028)
indices = [rng.randrange(len(differences)) for _ in differences]
one_difference = sum(differences[index] for index in indices) / len(indices)
print("One paired resample (percentage points):", 100 * one_difference)

In [ ]:
bootstrap_result = paired_bootstrap(baseline, candidate, draws=5000, seed=20261028)
point = 100 * bootstrap_result["difference"]
low = 100 * bootstrap_result["low"]
high = 100 * bootstrap_result["high"]
print(f"B−A: {point:+.1f} percentage points")
print(f"Central 95% percentile interval: [{low:+.1f}, {high:+.1f}]")
print("Draws:", bootstrap_result["draws"], "Seed:", bootstrap_result["seed"])

**Check:** the interval is `[−5, +13]` percentage points, so these items do not
establish a reliable positive gain. This does **not** prove that the models
are equivalent.

The interval conditions on two fixed fitted systems and prompts, assuming
independent items from the relevant population. It leaves out training seeds,
generation settings, prompt selection, and grader errors. Related rows may need
document- or task-level cluster resampling. The histogram in the slides uses
these exact samples; it is not a posterior probability that B is better.

To inspect the transparent implementation, open `assets/evaluation.py`.
Choose the next experiment according to the dominant uncertainty: additional
independent items and additional training runs answer different questions.

## Apply the method to your project

Draft a short evaluation record. This is practice, not an additional submission.

| Field | Your plan |
| --- | --- |
| Claim and baseline | |
| Data provenance, revision, and split boundary | |
| Model, tokenizer, prompt, and parser | |
| Sampling settings, tools, and attempt budget | |
| Metric, language/task slices, and aggregation | |
| Controlled comparison and resource cost | |
| Uncertainty estimate and failure inspection | |

Keep an independent final check after development choices. If your claim needs
GPU work, record memory per GPU and total GPU-hours across all devices/runs,
mark unmeasured costs as planning estimates, and preserve a smaller fallback.